# Securing Couchbase MCP Server with Keycloak — M2M Flow

This tutorial covers the **Machine-to-Machine (M2M) flow** for the Couchbase MCP server using Keycloak as the identity provider. In this flow, a client authenticates using a client ID and secret with no user login — ideal for automated agents, scripts, and server-to-server access.

---

## Prerequisites

- Docker installed (to run Keycloak locally)
- Couchbase MCP server installed via PyPI (`uvx couchbase-mcp-server`)
- (optional) MCP Inspector (`npx @modelcontextprotocol/inspector`) or any IDE (If IDE Authentication)
- A running Couchbase cluster with credentials
- `jq` installed for parsing JSON responses

---

## Part 1 — Run Keycloak Locally

```bash
docker run -p 8080:8080 \
  -e KC_BOOTSTRAP_ADMIN_USERNAME=admin \
  -e KC_BOOTSTRAP_ADMIN_PASSWORD=admin \
  quay.io/keycloak/keycloak:latest start-dev
```

Open `http://localhost:8080` and log in with `admin` / `admin`.


<img src="keycloak_screenshots/key_1.png" width="500">

---

## Part 2 — Keycloak Setup (shared by all flows)

### Step 2.1 — Create a Realm

1. Click the top-left dropdown → **Create Realm**
2. Name it `mcp-realm` → **Create**

<img src="keycloak_screenshots/key_2.png" width="500">

### Step 2.2 — Create custom scopes

1. Left nav → **Client Scopes** → **Create client scope**
2. Create `couchbase-mcp:read`:
   - **Name**: `couchbase-mcp:read`
   - **Type**: Optional
   - **Protocol**: openid-connect
   - **Include in token scope** → toggle **On** — this makes the scope name appear in the `scope` claim of the token. Without this, the scope is assigned but never shows up in the token even if the client requests it.
   - **Save**
3. Repeat for `couchbase-mcp:write`

<img src="keycloak_screenshots/key_3.png" width="500">

### Step 2.3 — Create the MCP Server client

This represents the resource server (the audience).

1. Left nav → **Clients** → **Create client**
2. **Client ID**: `couchbase-mcp-server`
3. **Client authentication**: Off (public)
4. Click through to **Save**
5. Go to **Client Scopes** tab → **Add client scope** → add both `couchbase-mcp:read` and `couchbase-mcp:write` as **Optional**


<img src="keycloak_screenshots/key_4.png" width="500">

<img src="keycloak_screenshots/key_5.png" width="500">

<img src="keycloak_screenshots/key_6.png" width="500">

### Step 2.4 — Add audience mapper

The MCP server checks the `aud` claim. You need to make Keycloak put `couchbase-mcp-server` in the token audience.

1. Go to **Client Scopes** → `couchbase-mcp:read` → **Mappers** tab → **Add mapper** → **By configuration** → **Audience**
2. Fill in:
   - **Name**: `couchbase-mcp-audience`
   - **Included Client Audience**: `couchbase-mcp-server`
   - **Add to access token**: On
3. **Save**
4. Repeat on `couchbase-mcp:write` scope (or add the mapper once to a shared scope)

<img src="keycloak_screenshots/key_10.png" width="500">


### Step 2.5 — Create a test user

1. Left nav → **Users** → **Add user**
2. Fill in:
   - **Username**: <USERNAME>
   - **First name**: <USER_FIRST_NAME>
   - **Last name**: <USER_LAST_NAME>
   - **Email**: <USER_EMAIL>
3. **Details** tab → toggle **Email verified** → **On** → **Save** — without this the password grant returns `Account is not fully set up`
4. **Credentials** tab → **Set password** → `password` → turn off **Temporary** → **Save**


<img src="keycloak_screenshots/key_11.png" width="500">

<img src="keycloak_screenshots/key_7.png" width="500">

---

## Part 3 — M2M Flow

This flow uses the password grant to get a static token on behalf of a user, then passes it as a Bearer token to the MCP server. Tested with MCP Inspector and a headless LangChain agent.

### Step 3.1 — Create a confidential client for token generation

1. Left nav → **Clients** → **Create client**
2. **Client ID**: `mcp-static-client`
3. **Client authentication**: On (confidential) — **important:** the Credentials tab only appears after Client authentication is enabled
4. **Authorization**: Off
5. **Authentication flow**: check **Direct access grants** — required for the password grant used to fetch tokens
6. **Save**
7. Go to **Credentials** tab → copy the **Client Secret**
8. **Client Scopes** tab → add `couchbase-mcp:read` and `couchbase-mcp:write` as Optional

<img src="keycloak_screenshots/key_8.png" width="500">

<img src="keycloak_screenshots/key_9.png" width="500">

### Step 3.2 — Get a static token

```bash
export TOKEN=$(curl -s -X POST \
  http://localhost:8080/realms/mcp-realm/protocol/openid-connect/token \
  -H "Content-Type: application/x-www-form-urlencoded" \
  -d "grant_type=password" \
  -d "client_id=mcp-static-client" \
  -d "client_secret=<YOUR_CLIENT_SECRET>" \
  -d "username=testuser" \
  -d "password=password" \
  -d "scope=openid couchbase-mcp:read couchbase-mcp:write" | jq -r .access_token)

echo "$TOKEN"
```

**Verify the token at [jwt.io](https://jwt.io)** — confirm:
- `iss` = `http://localhost:8080/realms/mcp-realm`
- `aud` contains `couchbase-mcp-server`
- `scope` contains `couchbase-mcp:read couchbase-mcp:write`

<img src="keycloak_screenshots/key_12.png" width="500">

### Step 3.3 — Start the MCP server

```bash
uvx couchbase-mcp-server \
  --transport=http \
  --connection-string="couchbase://127.0.0.1" \
  --username="Administrator" \
  --password="<your-couchbase-password>" \
  --read-only-mode=false \
  --oauth-jwks-uri="http://localhost:8080/realms/mcp-realm/protocol/openid-connect/certs" \
  --oauth-issuer="http://localhost:8080/realms/mcp-realm" \
  --oauth-audience="couchbase-mcp-server"
```

Note: **no `--oauth-mcp-base-url`** — pure token verification mode, no PRM endpoint needed for M2M.


**Verify the gate rejects unauthenticated calls:**

```bash
curl -i -X POST http://127.0.0.1:8000/mcp \
  -H "Content-Type: application/json" \
  -d '{"jsonrpc":"2.0","id":1,"method":"tools/list"}'
```

Expect a `401` response — confirms the OAuth gate is active.

### Step 3.4 — Test in MCP Inspector

```bash
npx @modelcontextprotocol/inspector
```

In the Inspector UI:
- **Transport Type**: Streamable HTTP
- **URL**: `http://127.0.0.1:8000/mcp`
- Under **Authentication** → **Custom Headers** → add:
  - Header name: `Authorization`
  - Header value: `Bearer <paste $TOKEN value>`
- Click **Connect**


**Verify:**
1. **Tools** tab → **List Tools** — should show all 24 Couchbase MCP tools
2. Run a read tool (e.g. `get_server_status`) → success
3. Run a write tool (e.g. `upsert_document_by_id`) → success (token has both scopes)

<img src="keycloak_screenshots/key_13.png" width="500">

### Step 3.5 — Configure VS Code `mcp.json`

```json
{
  "servers": {
    "couchbase-keycloak-static": {
      "type": "http",
      "url": "http://127.0.0.1:8000/mcp",
      "headers": {
        "Authorization": "Bearer <paste_access_token_here>"
      }
    }
  }
}
```

> ⚠️ **Token expiry:** Keycloak access tokens are short-lived (default 5 minutes). When it expires you'll see `401 Unauthorized`. Re-run Step 3.2 to get a fresh token and update `mcp.json`.

### Step 3.6 — Run as a headless LangChain agent

This is the production M2M pattern — a Python agent that fetches its own token and connects to the MCP server over HTTP with no user interaction, no browser, and no IDE.

The existing [LangChain + Couchbase MCP tutorial](https://developer.couchbase.com/tutorial-ai-agent-using-langchain-and-couchbase-mcp-server/) uses stdio transport without auth. This section adapts it to use **HTTP transport with OAuth M2M**.

#### Install dependencies

```bash
# Using uv (recommended)
mkdir mcp-agent-keycloak && cd mcp-agent-keycloak
uv init
uv add langchain langgraph langchain-openai langchain-mcp-adapters httpx python-dotenv
```

#### Set up environment variables

Create a `.env` file:

```bash
OPENAI_API_KEY=<your-openai-api-key>

# Keycloak M2M credentials
KEYCLOAK_BASE_URL=http://localhost:8080
KEYCLOAK_REALM=mcp-realm
KEYCLOAK_CLIENT_ID=mcp-static-client
KEYCLOAK_CLIENT_SECRET=<your-client-secret>
KEYCLOAK_USERNAME=testuser
KEYCLOAK_PASSWORD=password

# MCP server
MCP_SERVER_URL=http://127.0.0.1:8000/mcp
```

#### Agent code

Save as `agent.py`:

```python
import httpx
import os
import asyncio
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

load_dotenv()


def get_keycloak_token() -> str:
    """Fetch a token from Keycloak using password grant (M2M flow)."""
    response = httpx.post(
        f"{os.environ['KEYCLOAK_BASE_URL']}/realms/{os.environ['KEYCLOAK_REALM']}/protocol/openid-connect/token",
        data={
            "grant_type": "password",
            "client_id": os.environ["KEYCLOAK_CLIENT_ID"],
            "client_secret": os.environ["KEYCLOAK_CLIENT_SECRET"],
            "username": os.environ["KEYCLOAK_USERNAME"],
            "password": os.environ["KEYCLOAK_PASSWORD"],
            "scope": "openid couchbase-mcp:read couchbase-mcp:write",
        },
    )
    response.raise_for_status()
    return response.json()["access_token"]


async def run_agent(query: str):
    token = get_keycloak_token()

    client = MultiServerMCPClient(
        {
            "couchbase": {
                "url": os.environ["MCP_SERVER_URL"],
                "transport": "streamable_http",
                "headers": {"Authorization": f"Bearer {token}"},
            }
        }
    )
    tools = await client.get_tools()

    llm = ChatOpenAI(model="gpt-4o", temperature=0)
    agent = create_react_agent(llm, tools)

    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": query}]}
    )
    return result["messages"][-1].content


if __name__ == "__main__":
    answer = asyncio.run(
        run_agent("List all buckets in the Couchbase cluster")
    )
    print(answer)
```

#### Run it

Make sure the MCP server is running (Step 3.3), then:

```bash
uv run agent.py
```

The agent will:
1. Call Keycloak's token endpoint with the credentials → get a JWT automatically
2. Connect to the MCP server over HTTP with the JWT in the `Authorization` header
3. Discover the 24 Couchbase tools
4. Answer the query using LangChain's ReAct loop

<img src="keycloak_screenshots/key_14.png" width="500">

> ℹ️ **Why HTTP instead of stdio?** The original LangChain tutorial uses stdio transport — the agent spawns the MCP server as a subprocess. HTTP transport is needed here because OAuth token injection requires the client to control the HTTP headers, which stdio doesn't support. The MCP server runs as a separate process and the agent connects to it over HTTP with the Bearer token.

> ℹ️ **Token expiry:** Keycloak access tokens expire in 5 minutes by default (configurable in Realm Settings → Tokens). For long-running agents, call `get_keycloak_token()` before each session or implement token refresh logic.

